# 04b. Entity Resolution Profile Mart: multi-value витрина профилей

Этот ноутбук собирает профильную витрину для Entity Resolution, которая не теряет признаки с несколькими значениями у одного профиля.

Перед таблицами фиксируем термины:

- уровень события — одна исходная строка датасета, то есть конкретное наблюдение профиля в момент времени;
- уровень профиля — одна строка на `profile_id`;
- EAV — формат `entity / attribute / value`: объект, признак, значение;
- `block_rule` — имя правила группировки профилей. Чтобы не путать правило с исходным `feature_key`, все правила называем с префиксом `rule__`, например `rule__context__np_geoname_id` или `rule__identity_rescue__phone_digits`;
- `block_value` — конкретное значение правила, по которому профили попадают в один блок;
- many-to-many — один профиль может попасть в несколько блоков, и один блок может содержать несколько профилей.

Итоговые таблицы:

| Таблица | Уровень строки | Зачем нужна |
|---|---|---|
| `profile_core` | один `profile_id` | Базовый список профилей, число событий, даты первого/последнего события и label-derived поля для оценки качества. `entity_id` здесь только ground truth, не production-признак. |
| `profile_event_value_long` | одно значение признака в одном исходном событии | Полный аудит того, какие значения реально встречались в сырых данных. Нужна для проверки, отладки и пересборки агрегатов. |
| `profile_value_summary_long` | `profile_id + feature_key + value_norm` | Главный слой признаков с несколькими значениями: если у профиля было 7 разных значений `has_accept_365`, здесь будет 7 строк. |
| `profile_feature_stats_long` | `profile_id + feature_key` | Сводка по признаку внутри профиля: сколько наблюдений, сколько разных значений, какое значение основное. |
| `blocking_index` | `profile_id + block_rule + block_value` | Индекс кандидатов для сравнения профилей. Он показывает, по каким правилам и значениям профиль может быть сопоставлен с другими профилями. |



In [1]:
from pathlib import Path
import gc
import json
import hashlib
import re
import unicodedata

import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

## 1. Пути и настройки

`SAVE_EVENT_LEVEL_LONG=True` сохраняет полный EAV-слой уровня события. Он полезен для аудита, но файл крупнее. Для обычного анализа на уровне профиля достаточно `profile_value_summary_long`.

In [2]:
RAW_DATA_DIR = Path("../data/raw")
PROCESSED_DATA_DIR = Path("../data/processed")

INPUT_PATH = RAW_DATA_DIR / "split_label_train_V3.snappy.parquet"
OUTPUT_DIR = PROCESSED_DATA_DIR / "er_profile_mart_multivalue"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SAVE_EVENT_LEVEL_LONG = True

IDENTITY_COLS = ['first_name', 'last_name', 'email', 'phone', 'birthday', 'sex']

BASE_COLS = ['event_id', 'profile_id', 'entity_id', 'created_at']

INPUT_PATH, OUTPUT_DIR


(PosixPath('../data/raw/split_label_train_V3.snappy.parquet'),
 PosixPath('../data/processed/er_profile_mart_multivalue'))

## 2. Загрузка исходных событий

Исходная гранулярность: одна строка = одно событие / наблюдение профиля. Дальше агрегируем до уровня `profile_id`, но не теряем наборы значений.

In [3]:
raw = pd.read_parquet(INPUT_PATH).reset_index(drop=True)
# event_id позволяет понять, из какого конкретно события пришло
# значение;по нему считается value_count=('event_id', 'size');
# он помогает не потерять связь между raw-событием и
# распарсенными identity / np / rt / fs значениями;
raw['event_id'] = raw.index.astype('int64')
raw['created_at'] = pd.to_datetime(raw['created_at'], errors='coerce')

print('raw shape:', raw.shape)
print('n profiles:', raw['profile_id'].nunique())
print('n entities:', raw['entity_id'].nunique())
raw.head(1)

raw shape: (68036, 13)
n profiles: 61927
n entities: 53369


,created_at,first_name,last_name,email,phone,birthday,sex,non_processing_features,realtime_features,fs_features,profile_id,entity_id,event_id
0,2025-11-01 00:27:04.995,Анфиса,NaN,lqvxvltxx@mail.ru,NaN,None,female,"[device:smartphone, geoname_id:2013348, browse...","{""country"":""RU"",""is_million"":false,""tz_offset""...","[visited_30:250, visited_365:1813, visited_30:...",b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,0


## 3. Нормализация и парсинг

Разделяем:

- `value_raw_text` — исходное значение в parquet-safe текстовом виде;
- `value_norm` — нормализованное значение для анализа и blocking;
- `source` — семейство признаков: `identity`, `np`, `rt`, `fs`;
- `feature_key` — стабильное имя вида `source__feature`.

In [4]:
def norm_text(x):
    if pd.isna(x):
        return pd.NA
    s = unicodedata.normalize('NFKC', str(x)).strip().lower()
    s = re.sub(r'\s+', ' ', s)
    return s if s else pd.NA


def norm_phone(x):
    if pd.isna(x):
        return pd.NA
    digits = re.sub(r'\D+', '', str(x))
    # Простая нормализация для РФ: 8XXXXXXXXXX -> 7XXXXXXXXXX.
    if len(digits) == 11 and digits.startswith('8'):
        digits = '7' + digits[1:]
    return digits if digits else pd.NA


def norm_date_ymd(x):
    dt = pd.to_datetime(x, errors='coerce')
    if pd.isna(dt):
        return pd.NA
    return dt.strftime('%Y-%m-%d')


def normalize_identity_value(feature: str, value):
    if feature == 'phone':
        return norm_phone(value)
    if feature == 'birthday':
        return norm_date_ymd(value)
    return norm_text(value)


def parse_kv_token(token):
    # 'browser:chrome' -> ('browser', 'chrome')
    # 'is_phone'       -> ('is_phone', True)
    if token is None:
        return None, None
    s = str(token).strip()
    if not s:
        return None, None
    if ':' in s:
        key, value = s.split(':', 1)
        return key.strip(), value.strip()
    return s.strip(), True


def norm_feature_value(value):
    if value is True:
        return 'true'
    if value is False:
        return 'false'
    return norm_text(value)


def value_to_text(value):
    # Нужен единый тип для parquet: object-колонка с date/int/bool/string может падать при записи.
    if value is None or pd.isna(value):
        return pd.NA
    if isinstance(value, (dict, list, tuple)):
        return json.dumps(value, ensure_ascii=False, default=str)
    return str(value)

## 4. `profile_core`: базовая таблица профилей

Одна строка = один `profile_id`.  
`entity_id`, `entity_size`, `entity_kind` — это ground truth / label-derived metadata. Их можно использовать для оценки blocking и разметки пар, но нельзя использовать как production-признаки.

In [5]:
# profile_core отвечает на вопрос: “какие профили вообще есть и какая у них служебная информация”.
profile_core = (
    raw.groupby('profile_id', dropna=False, sort=False)
       .agg(
           event_count=('event_id', 'size'),
           entity_id=('entity_id', 'first'),
           entity_id_nunique=('entity_id', 'nunique'),
           first_event_at=('created_at', 'min'),
           last_event_at=('created_at', 'max'),
       )
       .reset_index()
)

entity_profile_counts = (
    raw.groupby('entity_id', sort=False)['profile_id']
       .nunique()
       .rename('entity_size')
       .reset_index()
)

profile_core = profile_core.merge(entity_profile_counts, on='entity_id', how='left')
profile_core['entity_kind'] = np.where(
    profile_core['entity_size'].fillna(0).astype(int) > 1,
    'multi_profile_entity',
    'single_profile_entity',
)

profile_core.to_parquet(OUTPUT_DIR / 'profile_core.parquet', index=False)
profile_core.head(1)

,profile_id,event_count,entity_id,entity_id_nunique,first_event_at,last_event_at,entity_size,entity_kind
0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,1,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,1,2025-11-01 00:27:04.995,2025-11-01 00:27:04.995,1,single_profile_entity


In [6]:
profile_core[profile_core["entity_size"] == 4].head(1)

,profile_id,event_count,entity_id,entity_id_nunique,first_event_at,last_event_at,entity_size,entity_kind
662,4198b906-8a45-3cb1-857c-0ee64c56f5ac,7,631e8407dd25b5085600f5b447a0326fdbfb773519809d...,1,2025-11-02 20:00:55.446,2026-04-08 21:42:33.769,4,multi_profile_entity


## 5. `profile_event_value_long`: EAV-слой уровня события

EAV = Entity–Attribute–Value.

Одна строка означает:

```text
в событии event_id у profile_id был признак feature_key со значением value_norm
```

Этот слой нужен для полного аудита. Если у профиля были разные `browser`, `geoname_id`, `source_site_365`, они остаются разными строками.

In [7]:
# 5.1 Identity-признаки
identity_parts = []

for col in IDENTITY_COLS:
    tmp = raw[BASE_COLS + [col]].rename(columns={col: 'value_raw'}).copy()
    tmp = tmp[tmp['value_raw'].notna()]
    if tmp.empty:
        continue

    tmp['source'] = 'identity'
    tmp['feature'] = col
    tmp['value_norm'] = tmp['value_raw'].map(lambda x, c=col: normalize_identity_value(c, x))
    tmp = tmp[tmp['value_norm'].notna()]
    identity_parts.append(tmp[BASE_COLS + ['source', 'feature', 'value_raw', 'value_norm']])

identity_event_long = (
    pd.concat(identity_parts, ignore_index=True)
    if identity_parts
    else pd.DataFrame(columns=BASE_COLS + ['source', 'feature', 'value_raw', 'value_norm'])
)

del identity_parts
gc.collect()

identity_event_long.head()

,event_id,profile_id,entity_id,created_at,source,feature,value_raw,value_norm
0,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,identity,first_name,Анфиса,анфиса
1,1,2403b42b-2420-35c2-ac8e-e6572c2920f6,bd0dc28d66040e40ad5303a3ed8f3988973726bb3d0307...,2025-11-01 00:38:35.323,identity,first_name,Надежда,надежда
2,3,70499ae9-52c6-3286-a763-de039d79533d,a6100b59b6ed947d8e7282f79d89297e7c93b5fe4ac057...,2025-11-01 00:50:56.909,identity,first_name,Альберт,альберт
3,4,845e13de-2ba8-361b-af0f-5c222bcdf417,f894d467f3e527b7f064f15aedcb5d24001fe1df707e0a...,2025-11-01 01:11:33.568,identity,first_name,Кира,кира
4,5,2c189c6d-3a9b-3706-b5d1-dbaebc2229a3,d26fef6237a85cd35fb7f42a65039e17ae62a79c5e4aaf...,2025-11-01 01:15:59.797,identity,first_name,Любовь,любовь


In [8]:
identity_event_long[identity_event_long["event_id"] == 7]

,event_id,profile_id,entity_id,created_at,source,feature,value_raw,value_norm
5,7,1a7c6db2-9e71-3d17-8b3e-8c9d6cdef6aa,f7ca9dbb24631ffa3d2f5e4c37876c5ff022f1d916641e...,2025-11-01 01:36:36.866,identity,first_name,Альберт,альберт
20107,7,1a7c6db2-9e71-3d17-8b3e-8c9d6cdef6aa,f7ca9dbb24631ffa3d2f5e4c37876c5ff022f1d916641e...,2025-11-01 01:36:36.866,identity,email,yvyltetdxmbdolgr51@gmail.com,yvyltetdxmbdolgr51@gmail.com
93516,7,1a7c6db2-9e71-3d17-8b3e-8c9d6cdef6aa,f7ca9dbb24631ffa3d2f5e4c37876c5ff022f1d916641e...,2025-11-01 01:36:36.866,identity,sex,unknown,unknown


In [9]:
# 5.2 non_processing_features и fs_features: списки token'ов вида key:value или flag
rows = []

for source_col, source_name in [('non_processing_features', 'np'), ('fs_features', 'fs')]:
    arr = raw[BASE_COLS + [source_col]].to_numpy(dtype=object)

    for event_id, profile_id, entity_id, created_at, values in arr:
        if values is None:
            continue

        for token in list(values):
            feature, value = parse_kv_token(token)
            if feature is None:
                continue

            value_norm = norm_feature_value(value)
            if pd.isna(value_norm):
                continue

            rows.append((event_id, profile_id, entity_id, created_at, source_name, feature, value, value_norm))

list_event_long = pd.DataFrame.from_records(
    rows,
    columns=BASE_COLS + ['source', 'feature', 'value_raw', 'value_norm'],
)

del rows
gc.collect()

list_event_long.head(2)

,event_id,profile_id,entity_id,created_at,source,feature,value_raw,value_norm
0,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,np,device,smartphone,smartphone
1,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,np,geoname_id,2013348,2013348


In [10]:
# 5.3 realtime_features: JSON-объект
rows = []

arr = raw[BASE_COLS + ['realtime_features']].to_numpy(dtype=object)

for event_id, profile_id, entity_id, created_at, payload in arr:
    if payload is None or pd.isna(payload):
        continue

    try:
        obj = json.loads(payload)
    except Exception:
        continue

    if not isinstance(obj, dict):
        continue

    for feature, value in obj.items():
        value_norm = norm_feature_value(value)
        if pd.notna(value_norm):
            rows.append((event_id, profile_id, entity_id, created_at, 'rt', str(feature), value, value_norm))

realtime_event_long = pd.DataFrame.from_records(
    rows,
    columns=BASE_COLS + ['source', 'feature', 'value_raw', 'value_norm'],
)

del rows
gc.collect()

realtime_event_long.head(2)

,event_id,profile_id,entity_id,created_at,source,feature,value_raw,value_norm
0,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,rt,country,RU,ru
1,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,rt,is_million,False,false


In [11]:
profile_event_value_long = pd.concat(
    [identity_event_long, list_event_long, realtime_event_long],
    ignore_index=True,
)

del identity_event_long, list_event_long, realtime_event_long
gc.collect()

profile_event_value_long['value_raw_text'] = profile_event_value_long['value_raw'].map(value_to_text)
profile_event_value_long = profile_event_value_long.drop(columns=['value_raw'])
profile_event_value_long['feature_key'] = (
    profile_event_value_long['source'].astype(str) + '__' + profile_event_value_long['feature'].astype(str)
)

# Категории ускоряют groupby и уменьшают память.
for col in ['profile_id', 'source', 'feature', 'feature_key', 'value_norm']:
    profile_event_value_long[col] = profile_event_value_long[col].astype('category')

print('profile_event_value_long shape:', profile_event_value_long.shape)
profile_event_value_long.head()

profile_event_value_long shape: (2110680, 9)


,event_id,profile_id,entity_id,created_at,source,feature,value_norm,value_raw_text,feature_key
0,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,identity,first_name,анфиса,Анфиса,identity__first_name
1,1,2403b42b-2420-35c2-ac8e-e6572c2920f6,bd0dc28d66040e40ad5303a3ed8f3988973726bb3d0307...,2025-11-01 00:38:35.323,identity,first_name,надежда,Надежда,identity__first_name
2,3,70499ae9-52c6-3286-a763-de039d79533d,a6100b59b6ed947d8e7282f79d89297e7c93b5fe4ac057...,2025-11-01 00:50:56.909,identity,first_name,альберт,Альберт,identity__first_name
3,4,845e13de-2ba8-361b-af0f-5c222bcdf417,f894d467f3e527b7f064f15aedcb5d24001fe1df707e0a...,2025-11-01 01:11:33.568,identity,first_name,кира,Кира,identity__first_name
4,5,2c189c6d-3a9b-3706-b5d1-dbaebc2229a3,d26fef6237a85cd35fb7f42a65039e17ae62a79c5e4aaf...,2025-11-01 01:15:59.797,identity,first_name,любовь,Любовь,identity__first_name


In [12]:

profile_event_value_long[profile_event_value_long["event_id"] == 0]

,event_id,profile_id,entity_id,created_at,source,feature,value_norm,value_raw_text,feature_key
0,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,identity,first_name,анфиса,Анфиса,identity__first_name
20100,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,identity,email,lqvxvltxx@mail.ru,lqvxvltxx@mail.ru,identity__email
93509,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,identity,sex,female,female,identity__sex
161493,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,np,device,smartphone,smartphone,np__device
161494,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,np,geoname_id,2013348,2013348,np__geoname_id
161495,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,np,browser,chrome,chrome,np__browser
161496,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,np,osfamily,android,android,np__osfamily
161497,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,np,subdivision_1_iso_code,pri,PRI,np__subdivision_1_iso_code
471120,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,fs,visited_30,250,250,fs__visited_30
471121,0,b25135c7-b4f8-3552-8eb6-46331ceea5c9,7266581b455c33f90f4e2418ab7d12e3ec704935338ecf...,2025-11-01 00:27:04.995,fs,visited_365,1813,1813,fs__visited_365


## 6. Словарь признаков

Быстрый обзор того, какие признаки получились после разворачивания списков и JSON.

In [13]:
feature_dictionary = (
    profile_event_value_long
    .groupby(['source', 'feature', 'feature_key'], observed=True, sort=False)
    .agg(
        token_count=('event_id', 'size'),
        n_profiles=('profile_id', 'nunique'),
        n_values=('value_norm', 'nunique'),
    )
    .reset_index()
)

feature_dictionary['profile_share'] = feature_dictionary['n_profiles'] / raw['profile_id'].nunique()
feature_dictionary = feature_dictionary.sort_values(['source', 'n_profiles'], ascending=[True, False])
feature_dictionary.to_csv(OUTPUT_DIR / 'feature_dictionary.csv', index=False)

feature_dictionary.head(1)

,source,feature,feature_key,token_count,n_profiles,n_values,profile_share
14,fs,source_site_365,fs__source_site_365,84173,31684,335,0.511635


## 7. `profile_value_summary_long`: главный multi-value слой

Одна строка означает:

```text
у profile_id для feature_key встречалось value_norm N раз
```

Если у профиля было несколько значений одного признака, они остаются несколькими строками. Именно эта таблица — источник истины для multi-value анализа.

In [14]:
profile_value_summary_long = (
    profile_event_value_long
    .groupby(['profile_id', 'source', 'feature', 'feature_key', 'value_norm'], observed=True, sort=False)
    .agg(
        value_count=('event_id', 'size'),
        first_seen=('created_at', 'min'),
        last_seen=('created_at', 'max'),
    )
    .reset_index()
)

first_raw = (
    profile_event_value_long
    .drop_duplicates(['profile_id', 'feature_key', 'value_norm'])
    [['profile_id', 'feature_key', 'value_norm', 'value_raw_text']]
    .rename(columns={'value_raw_text': 'value_raw_example'})
)

profile_value_summary_long = profile_value_summary_long.merge(
    first_raw,
    on=['profile_id', 'feature_key', 'value_norm'],
    how='left',
)

del first_raw
gc.collect()

# feature_observation_count
# Это общее число наблюдений конкретного признака у профиля.
feature_obs = (
    profile_value_summary_long
    .groupby(['profile_id', 'feature_key'], observed=True, sort=False)['value_count']
    .sum()
    .rename('feature_observation_count')
    .reset_index()
)

profile_value_summary_long = profile_value_summary_long.merge(
    feature_obs,
    on=['profile_id', 'feature_key'],
    how='left',
)

del feature_obs
gc.collect()
# value_share_within_feature
# Это доля конкретного значения внутри всех наблюдений
profile_value_summary_long['value_share_within_feature'] = (
    profile_value_summary_long['value_count'] / profile_value_summary_long['feature_observation_count']
)

profile_value_summary_long = profile_value_summary_long.sort_values(
    ['profile_id', 'feature_key', 'value_count', 'first_seen', 'value_norm'],
    ascending=[True, True, False, True, True],
)
# value_rank Ранг значения внутри признака у профиля.
# То есть value_rank = 1 — самое частое значение. Если частоты
# равны, берём более раннее, потом
# лексикографически меньшее для стабильности.
# Итого, это самое часто встречавшееся значение конкретного признака внутри конкретного профиля.
profile_value_summary_long['value_rank'] = (
    profile_value_summary_long
    .groupby(['profile_id', 'feature_key'], observed=True)
    .cumcount() + 1
)

# is_primary_value Флаг: is_primary_value = value_rank == 1 То есть это выбранное главное значение признака
# у профиля. Важно: это не замена остальных значений
profile_value_summary_long['is_primary_value'] = profile_value_summary_long['value_rank'].eq(1)

profile_value_summary_long.to_parquet(OUTPUT_DIR / 'profile_value_summary_long.parquet', index=False)

print('profile_value_summary_long shape:', profile_value_summary_long.shape)
profile_value_summary_long.head(3)

profile_value_summary_long shape: (1844644, 13)


,profile_id,source,feature,feature_key,value_norm,value_count,first_seen,last_seen,value_raw_example,feature_observation_count,value_share_within_feature,value_rank,is_primary_value
79672,000163e0-b9b4-3323-84ea-6495df7fe895,identity,email,identity__email,srnxztxqa8004@gmail.com,1,2026-05-09 07:47:48.739,2026-05-09 07:47:48.739,srnxztxqa8004@gmail.com,1,1.0,1,True
146579,000163e0-b9b4-3323-84ea-6495df7fe895,identity,sex,identity__sex,unknown,1,2026-05-09 07:47:48.739,2026-05-09 07:47:48.739,unknown,1,1.0,1,True
428406,000163e0-b9b4-3323-84ea-6495df7fe895,np,browser,np__browser,chrome,1,2026-05-09 07:47:48.739,2026-05-09 07:47:48.739,chrome,1,1.0,1,True


In [15]:
profile_value_summary_long[profile_value_summary_long["profile_id"] ==
 "0001ced1-cecc-3dc9-b4c1-acf162c8b2b5"].head(1)
#profile_value_summary_long[profile_value_summary_long["is_primary_value"] == "True"]["value_raw_example"]
# .nunique()
#profile_value_summary_long[profile_value_summary_long["value_rank"] > 1]

,profile_id,source,feature,feature_key,value_norm,value_count,first_seen,last_seen,value_raw_example,feature_observation_count,value_share_within_feature,value_rank,is_primary_value
750534,0001ced1-cecc-3dc9-b4c1-acf162c8b2b5,fs,has_accept_365,fs__has_accept_365,3824,1,2026-01-26 13:13:07.558,2026-01-26 13:13:07.558,3824,4,0.25,1,True


In [16]:
if SAVE_EVENT_LEVEL_LONG:
    profile_event_value_long.to_parquet(OUTPUT_DIR / 'profile_event_value_long.parquet', index=False)
else:
    print('Skipped saving long table at event level. Set SAVE_EVENT_LEVEL_LONG=True if full event audit is needed.')

# Keep profile_event_value_long in memory:
# Rows at event level are needed by profile_feature_stats_long and for interactive reruns.


## 8. `profile_feature_stats_long`: статистика признака внутри профиля

Этот слой отвечает на вопросы:

- сколько value-observations признака было у профиля;
- в скольких raw-событиях признак присутствовал;
- сколько уникальных значений было у признака;
- какое значение выбрано как `primary_value`;
- является ли признак multi-value внутри профиля.

Важно: для list/json признаков одно raw-событие может дать несколько значений одного `feature_key`. Поэтому разделяем две разные метрики:

```text
feature_value_count             # сколько значений/токенов признака наблюдали
events_with_feature_count       # в скольких raw-событиях признак присутствовал
feature_event_coverage          # events_with_feature_count / event_count
feature_observations_per_event  # feature_value_count / event_count
```

`primary_value` — не потеря остальных значений, а только удобное представительное значение. Полный набор остаётся в `profile_value_summary_long`.


In [17]:
feature_event_counts = (
    profile_event_value_long
    .drop_duplicates(['profile_id', 'feature_key', 'event_id'])
    .groupby(['profile_id', 'feature_key'], observed=True, sort=False)['event_id']
    .size()
    .rename('events_with_feature_count')
    .reset_index()
)

profile_feature_stats_long = (
    profile_value_summary_long
    .groupby(['profile_id', 'source', 'feature', 'feature_key'], observed=True, sort=False)
    .agg(
        feature_value_count=('value_count', 'sum'),
        feature_nunique=('value_norm', 'nunique'),
        primary_value=('value_norm', 'first'),
        primary_value_count=('value_count', 'first'),
        first_seen=('first_seen', 'min'),
        last_seen=('last_seen', 'max'),
    )
    .reset_index()
)

profile_feature_stats_long = profile_feature_stats_long.merge(
    feature_event_counts,
    on=['profile_id', 'feature_key'],
    how='left',
)

# True when a profile has more than one unique normalized value for this feature.
profile_feature_stats_long['is_multivalue'] = profile_feature_stats_long['feature_nunique'].gt(1)

profile_feature_stats_long = profile_feature_stats_long.merge(
    profile_core[['profile_id', 'event_count']],
    on='profile_id',
    how='left',
)

profile_feature_stats_long['events_with_feature_count'] = (
    profile_feature_stats_long['events_with_feature_count'].fillna(0).astype('int64')
)
profile_feature_stats_long['feature_event_coverage'] = (
    profile_feature_stats_long['events_with_feature_count'] / profile_feature_stats_long['event_count']
)
profile_feature_stats_long['feature_observations_per_event'] = (
    profile_feature_stats_long['feature_value_count'] / profile_feature_stats_long['event_count']
)

profile_feature_stats_long.to_parquet(OUTPUT_DIR / 'profile_feature_stats_long.parquet', index=False)

# Keep profile_event_value_long in memory so this cell can be rerun interactively.
del feature_event_counts
gc.collect()

print('profile_feature_stats_long shape:', profile_feature_stats_long.shape)
profile_feature_stats_long.head(2)


profile_feature_stats_long shape: (1199575, 15)


,profile_id,source,feature,feature_key,feature_value_count,feature_nunique,primary_value,primary_value_count,first_seen,last_seen,events_with_feature_count,is_multivalue,event_count,feature_event_coverage,feature_observations_per_event
0,000163e0-b9b4-3323-84ea-6495df7fe895,identity,email,identity__email,1,1,srnxztxqa8004@gmail.com,1,2026-05-09 07:47:48.739,2026-05-09 07:47:48.739,1,False,1,1.0,1.0
1,000163e0-b9b4-3323-84ea-6495df7fe895,identity,sex,identity__sex,1,1,unknown,1,2026-05-09 07:47:48.739,2026-05-09 07:47:48.739,1,False,1,1.0,1.0


Интерпретация:
У профиля всего 29 raw-событий.

Во всех 29 событиях встречался хотя бы один token has_accept_365:...

Всего после explode было найдено 186 token-ов has_accept_365:...

Среди этих 186 token-ов было 7 разных значений после двоеточия.

В среднем на одно raw-событие:
186 / 29 = 6.41 token-ов

In [18]:
profile_feature_stats_long[profile_feature_stats_long["profile_id"] ==
                            "0765344e-20aa-3f9d-9a69-66eb41375af4"].head(3)

,profile_id,source,feature,feature_key,feature_value_count,feature_nunique,primary_value,primary_value_count,first_seen,last_seen,events_with_feature_count,is_multivalue,event_count,feature_event_coverage,feature_observations_per_event
35128,0765344e-20aa-3f9d-9a69-66eb41375af4,fs,has_accept_30,fs__has_accept_30,105,5,3684,29,2025-11-14 04:14:10.456,2026-02-26 06:58:55.574,29,True,29,1.0,3.620690
35129,0765344e-20aa-3f9d-9a69-66eb41375af4,fs,has_accept_365,fs__has_accept_365,186,7,3684,29,2025-11-14 04:14:10.456,2026-02-26 06:58:55.574,29,True,29,1.0,6.413793
35130,0765344e-20aa-3f9d-9a69-66eb41375af4,fs,has_account,fs__has_account,174,6,1345,29,2025-11-14 04:14:10.456,2026-02-26 06:58:55.574,29,True,29,1.0,6.000000


## 9. `blocking_index`: поведенческий и контекстный blocking

Это ещё не candidate pairs и не решение о дублях.

`blocking_index` хранит many-to-many индекс:

```text
profile_id → block_family → block_rule → block_value
```

Главная идея: не строить blocking вокруг разреженных identity-полей. `email` почти уникален, `phone`, `last_name`, `birthday` заполнены слабо, поэтому они плохо подходят как основа группировки.

`block_family` — это тип или роль blocking-правила. Он нужен, чтобы не смешивать в одну кучу разные по смыслу правила.

В этом разделе используем семь семейств:

- `context` — контекстные ключи, в основном география: `rule__context__np_geoname_id`, `rule__context__np_subdivision`, `rule__context__rt_geoid`, `rule__context__rt_geoname`. Это основа для высокого охвата настоящих дублей, потому что гео-признаки заполнены существенно лучше identity-полей.
- `behavior` — поведенческие `fs` / site-id признаки: `rule__behavior__fs_source_site_365`, `rule__behavior__fs_source_site_30`, `rule__behavior__fs_visited_30`, `rule__behavior__fs_has_account` и другие. Они отражают поведенческий след профиля; совпадение по редкому site-id часто полезнее, чем совпадение по имени.
- `behavior_context` — композитные блоки вида география + поведенческий признак: например `rule__behavior_context__np_geoname_id__fs_source_site_365`, `rule__behavior_context__np_geoname_id__fs_has_click_365`. Такие блоки обычно чище: меньше случайных совпадений и меньше candidate pairs.
- `behavior_context_device` — более узкие rescue-композиты вида регион + поведенческий признак + OS family. Их добавили после rule discovery и deep-dive анализа: они дают небольшой marginal recall сверх текущих правил, но шумят меньше, чем широкие behavior-only или browser-правила.
- `coverage_compound` — дополнительные композиты для профилей, которые не покрывались основной схемой: `geo + device/osfamily`, `email_domain + device/osfamily/sex`, `email_domain + email_initial2`. Это coverage-слой, но часть правил ещё и заметно повышает recall.
- `coverage_fallback` — последний технический fallback `rule__coverage_fallback__email_hash_bucket_1024`. Он гарантирует, что каждый профиль попадёт хотя бы в один блок размера `2..1000`. Это не сильный ER-сигнал, а защита от потери профилей на этапе candidate generation.
- `identity_rescue` — точечные телефонные правила: `rule__identity_rescue__phone_digits`, `rule__identity_rescue__phone_prefix6`. Это не основа blocking, а способ не потерять редкие, но сильные совпадения по телефону.

Ограничение размера блока не зашиваем в название правила. Оно задаётся параметрами `BLOCK_MIN_SIZE = 2` и `BLOCK_MAX_SIZE = 1000`. Поэтому `rule__behavior__fs_has_account` — это именно правило blocking, а `fs__has_account` — это исходный признак в витрине.

Что происходит с остальными блоками:

- блоки размера `1` отбрасываются, потому что внутри них нет пары профилей для сравнения;
- блоки размера больше `1000` отбрасываются, потому что они создают слишком много шумных candidate pairs;
- такие значения не попадают в `blocking_index`, но сами признаки остаются в витрине и могут использоваться
 позже как similarity-признаки при сравнении уже найденных candidate pairs. А именно если для конкретного
 block_value число уникальных profile_id < 2 или > 1000, то этот block_value не используется в blocking_index

`block_family` — не фича профиля. Это метка самого blocking-правила, чтобы понимать, какую роль оно играет в генерации candidate pairs.

Финальная проверка в этом ноутбуке требует `profile_coverage = 1.0` и для полного `blocking_index`, и для правил `recommended_for_next_step`, которые дальше использует baseline-модель.

Identity-признаки дальше лучше использовать как similarity/conflict features при сравнении candidate pairs.

In [19]:
pv = profile_value_summary_long

BLOCK_MIN_SIZE = 2
BLOCK_MAX_SIZE = 1000


def stable_hash_bucket(value, n_buckets: int) -> str:
    value = '' if pd.isna(value) else str(value)
    bucket = int(hashlib.md5(value.encode('utf-8')).hexdigest()[:8], 16) % n_buckets
    return str(bucket)


def derived_feature_values(feature: str, colname: str) -> pd.DataFrame:
    emails = pv[(pv['source'] == 'identity') & (pv['feature'] == 'email')][['profile_id', 'value_norm']].copy()
    emails['email_norm'] = emails['value_norm'].astype(str).str.lower()
    emails['email_domain'] = emails['email_norm'].str.extract(r'@([^@]+)$', expand=False).fillna('missing_domain')
    emails['email_local'] = emails['email_norm'].str.extract(r'^([^@]+)', expand=False).fillna('missing_local')
    emails['email_initial1'] = emails['email_local'].str[:1].replace('', 'missing')
    emails['email_initial2'] = emails['email_local'].str[:2].replace('', 'missing')
    emails['email_hash_bucket_1024'] = emails['email_local'].map(lambda value: stable_hash_bucket(value, 1024))

    if feature not in emails.columns:
        return pd.DataFrame(columns=['profile_id', colname])

    out = emails[['profile_id', feature]].rename(columns={feature: colname})
    return out[['profile_id', colname]].drop_duplicates()


def feature_values(source: str, feature: str, colname: str) -> pd.DataFrame:
    if source == 'derived':
        return derived_feature_values(feature, colname)

    sub = pv[(pv['source'] == source) & (pv['feature'] == feature)][['profile_id', 'value_norm']].copy()
    sub[colname] = sub['value_norm'].astype(str)
    return sub[['profile_id', colname]].drop_duplicates()

# Размер блока — это сколько уникальных profile_id имеют одно
# то же block_value внутри конкретного block_rule.
def apply_block_size_limits(
    sub: pd.DataFrame,
    block_value_col: str = 'block_value',
    min_block_size: int = BLOCK_MIN_SIZE,
    max_block_size: int = BLOCK_MAX_SIZE,
) -> pd.DataFrame:
    if sub.empty:
        return sub

    sizes = (
        sub.groupby(block_value_col, sort=False)['profile_id']
        .nunique()
        .rename('block_size')
        .reset_index()
    )
    keep_values = sizes[
        sizes['block_size'].between(min_block_size, max_block_size)
    ][[block_value_col]]

    return sub.merge(keep_values, on=block_value_col, how='inner')


def atomic_block(
    rule: str,
    source: str,
    feature: str,
    family: str,
    transform=None,
    min_len: int = 1,
    min_block_size: int = BLOCK_MIN_SIZE,
    max_block_size: int = BLOCK_MAX_SIZE,
) -> pd.DataFrame:
    """Создать blocking-правило по одному признаку.

    Пример: `rule__context__np_geoname_id` берёт все значения признака
    `source='np', feature='geoname_id'` из `profile_value_summary_long`.

    Для каждого профиля получается строка вида:
    `profile_id + block_family + block_rule + block_value`.

    Дальше применяются ограничения:
    - пустые значения удаляются;
    - слишком короткие значения удаляются через `min_len`;
    - блоки размера меньше `min_block_size` и больше `max_block_size` не попадают в индекс.

    Важно: `block_rule` — это имя правила, а не имя исходного признака.
    Поэтому используем префикс `rule__`, чтобы не путать правило с `feature_key`.
    """
    sub = feature_values(source, feature, 'block_value')
    if sub.empty:
        return pd.DataFrame(columns=['profile_id', 'block_family', 'block_rule', 'block_value'])

    if transform is not None:
        sub['block_value'] = sub['block_value'].map(transform)

    sub = sub[sub['block_value'].notna()]
    sub = sub[sub['block_value'].astype(str).str.len().ge(min_len)]
    sub = apply_block_size_limits(sub, 'block_value', min_block_size, max_block_size)
    sub['block_family'] = family
    sub['block_rule'] = rule

    return sub[['profile_id', 'block_family', 'block_rule', 'block_value']].drop_duplicates()


def composite_block(
    rule: str,
    specs: list[tuple],
    family: str,
    sep: str = '|',
    min_block_size: int = BLOCK_MIN_SIZE,
    max_block_size: int = BLOCK_MAX_SIZE,
) -> pd.DataFrame:
    """Создать blocking-правило по комбинации нескольких признаков.

    `specs` — список компонентов блока в формате:
    `(source, feature, output_colname, transform, min_len)`.

    Пример:
    `np.subdivision_1_iso_code + np.device + np.osfamily`.

    Как работает сборка:
    1. Для каждого компонента берём таблицу `profile_id + значение`.
    2. Удаляем пустые и слишком короткие значения.
    3. Склеиваем компоненты через inner join по `profile_id`.
       Значит, профиль попадёт в композитный блок только если у него есть все компоненты.
    4. Собираем `block_value` как строку из компонент через `sep`, например:
       `RU-MOW|smartphone|android`.
    5. Ограничиваем размер блока диапазоном `min_block_size..max_block_size`.
    6. Возвращаем строки `profile_id + block_family + block_rule + block_value`.

    Композит обычно точнее атомарного правила: совпадение только по региону слабое,
    а совпадение по региону + устройству + OS family уже заметно сужает область поиска.

    Например если у профиля есть:
        np.subdivision_1_iso_code = A, B
        np.device = mobile, desktop
        np.osfamily = android
        то composite_block() через inner join создаст комбинации:
        A|mobile|android
        A|desktop|android
        B|mobile|android
        B|desktop|android
        То есть профиль попадёт сразу в несколько composite block values.
        Это сделано в пользу recall: мы не теряем дубль только потому, что “главное” значение выбрали не то. Но минус — больше candidate pairs и больше шума.

    """
    # specs: (source, feature, output_colname, transform, min_len)
    out = None
    cols = []

    for source, feature, colname, transform, min_len in specs:
        vals = feature_values(source, feature, colname)

        if transform is not None:
            vals[colname] = vals[colname].map(transform)

        vals = vals[vals[colname].notna()]
        vals = vals[vals[colname].astype(str).str.len().ge(min_len)]

        out = vals if out is None else out.merge(vals, on='profile_id', how='inner')
        cols.append(colname)

    if out is None or out.empty:
        return pd.DataFrame(columns=['profile_id', 'block_family', 'block_rule', 'block_value'])

    out['block_value'] = out[cols].astype(str).agg(sep.join, axis=1)
    out = apply_block_size_limits(out[['profile_id', 'block_value']], 'block_value', min_block_size, max_block_size)
    out['block_family'] = family
    out['block_rule'] = rule

    return out[['profile_id', 'block_family', 'block_rule', 'block_value']].drop_duplicates()


prefix6 = lambda s: s[:6] if isinstance(s, str) and len(s) >= 6 else None


## Алгоритм создания первого слоя правил blocking

Первый слой правил нужен не для финального решения “это один клиент или нет”, а для генерации разумного списка candidate pairs.
Идея: вместо сравнения всех профилей со всеми мы группируем профили по нескольким признакам и сравниваем только тех, кто попал в один блок.


### Общий алгоритм

1. Берём значение признака из `profile_value_summary_long`.

2. Нормализуем значение, если нужно:
   - приводим к строке;
   - убираем пустые значения;
   - для телефона можем оставить только цифры;
   - для email можем выделить домен, первые символы local-part или hash bucket.

3. Строим блок:
   - для атомарного правила: один признак → один `block_value`;
   - для композитного правила: несколько признаков склеиваются в один `block_value`.

Пример атомарного правила:

```text
rule__context__np_geoname_id
profile_id = P1, np.geoname_id = 524901
→ block_value = 524901
```

Пример композитного правила:

```python
composite_block(
    'rule__coverage__np_subdivision__np_device__np_osfamily',
    [
        ('np', 'subdivision_1_iso_code', 'subdivision', None, 2),
        ('np', 'device', 'device', None, 1),
        ('np', 'osfamily', 'osfamily', None, 1),
    ],
    family='coverage_compound',
    max_block_size=1000,
)
```

Здесь сборка работает так:

1. Берём для каждого профиля значение `np.subdivision_1_iso_code`.
2. Берём для этого же профиля значение `np.device`.
3. Берём для этого же профиля значение `np.osfamily`.
4. Оставляем только профили, у которых есть все три компонента.
5. Склеиваем значения в один ключ, например:

```text
subdivision=RU-MOW
device=smartphone
osfamily=android
→ block_value = RU-MOW|smartphone|android
```

6. Все профили с таким же `block_value` попадают в один блок.
7. Если в блоке меньше 2 профилей или больше 1000 профилей, блок не попадает в `blocking_index`.

То есть композитный блок не хранит отдельные признаки по отдельности. Он создаёт новый группировочный ключ из комбинации признаков.

4. Считаем размер каждого блока:
- сколько уникальных `profile_id` попало в один `block_value`.

5. Применяем ограничение размера блока:
   - блоки размера `1` отбрасываем, потому что внутри них нет пары для сравнения;
   - слишком большие блоки отбрасываем, потому что они создают много случайных пар;
   - сейчас используем диапазон `2..1000`.

6. Добавляем служебные поля:
   - `block_family`;
   - `block_rule`;
   - `block_value`.

7. Объединяем все правила в один `blocking_index`.

8. Удаляем дубли вида:
   - один и тот же `profile_id`;
   - одно и то же правило;
   - одно и то же значение блока.

9. Проверяем покрытие:
   - полный `blocking_index` должен покрывать 100% профилей;
   - набор правил, рекомендованный для следующего шага, тоже должен покрывать 100% профилей.

### Как выбираем правила

Правила делятся на несколько семейств.

| Семейство | Роль |
|---|---|
| `context` | Широкий слой охвата: география и контекст. Даёт recall, но может шуметь. |
| `behavior` | Поведенческие `fs` / site-id признаки. Часто дают полезный сигнал, особенно если значение редкое. |
| `behavior_context` | Композиты вида география + поведенческий признак. Обычно чище, чем одиночный context или behavior. |
| `behavior_context_device` | Узкие композиты вида регион + поведение + устройство/OS. Используются как аккуратный дополнительный слой. |
| `identity_rescue` | Редкие, но сильные identity-совпадения, например телефон. Не основа blocking, а rescue-правила. |
| `coverage_compound` | Дополнительные композиты для покрытия профилей, которые плохо ловятся основными правилами. |
| `coverage_fallback` | Последний технический слой, чтобы не потерять профиль полностью. Это слабый сигнал. |

### Почему используем именно эти правила

Правила выбраны не вручную “с потолка”. Мы опирались на предыдущий EDA, анализ заполненности identity-полей, rule diagnostics, deep-dive по реальным дублям и анализ ложных склеек в графе.

| Правило / группа правил | Почему добавляем | Риск | Как ограничиваем риск |
|---|---|---|---|
| `rule__context__np_geoname_id`, `rule__context__np_subdivision`, `rule__context__rt_geoid`, `rule__context__rt_geoname` | Гео/context признаки хорошо заполнены и дают широкий recall: многие реальные дубли совпадают хотя бы по географии. | Гео само по себе слабое: в одном городе/регионе много разных клиентов. | Ограничиваем размер блока и дальше не считаем context-only совпадение достаточным сильным evidence. |
| `rule__behavior__fs_*` | Поведенческие site-id признаки часто сужают поиск лучше, чем имя или email-domain: совпадение по редкому site-id может быть полезным сигналом. | Частые site-id дают шумные блоки. | Отбрасываем слишком большие блоки; дальше учитываем размер блока и число сработавших правил. |
| `rule__behavior_context__np_geoname_id__fs_*` | Комбинация географии и поведения обычно чище, чем каждый признак отдельно: кандидат должен совпасть и по месту, и по поведению. | Можно потерять часть дублей, если один из компонентов отсутствует. | Используем как дополнительный слой, а не единственный способ найти пару. |
| `rule__behavior_context_device__np_subdivision__fs_*__np_osfamily` | Эти узкие правила появились после анализа реальных дублей и rule discovery: они дают небольшой дополнительный recall при умеренном шуме. | Слишком узкое правило может покрывать мало профилей. | Оставляем только несколько проверенных `fs`-признаков, не расширяем на все подряд. |
| `rule__identity_rescue__phone_digits`, `rule__identity_rescue__phone_prefix6` | Телефон заполнен редко, но когда есть, это один из самых сильных identity-сигналов. | Телефона мало, поэтому он не может быть основой blocking. Префикс телефона слабее полного номера. | Используем как rescue-слой, а не backbone. Полный номер сильнее, префикс нужен только как дополнительный шанс не потерять пару. |
| `rule__coverage__np_geoname_id__np_device__np_osfamily`, `rule__coverage__np_subdivision__np_device__np_osfamily` | Нужны для профилей, которые не попадают в более сильные behavior-композиты. Это попытка сохранить покрытие через context + device. | В анализе ложных склеек такие правила часто шумели, если использовать их как самостоятельный сильный сигнал. | Помечаем семейством `coverage_compound`; на graph/model layer такие правила считаются слабыми и требуют дополнительного evidence. |
| `rule__coverage__email_domain__np_device__np_osfamily__sex`, `rule__coverage__email_domain__email_initial2` | Email заполнен почти полностью, но полный email почти уникален. Поэтому используем не сам email, а более грубые производные для coverage. | Email-domain и initials слишком общие, могут создавать случайные совпадения. | Используем только как coverage-слой, не как high-precision identity-сигнал. |
| `rule__coverage_fallback__email_hash_bucket_1024` | Технический fallback, чтобы каждый профиль имел хотя бы один blocking-key и не выпадал из процесса полностью. | Это слабое искусственное правило, оно может создавать случайные пары. | Отдельное семейство `coverage_fallback`; дальше такие пары считаются fallback-only и фильтруются/штрафуются. |

Ключевая идея: первый слой правил должен давать покрытие и не потерять потенциальные дубли. Точность переносится на следующий слой: pair evidence, graph/tree edge policy и анализ ложных склеек.



In [20]:
# Context rules: широкий recall-слой по географии/context.
# Они хорошо покрывают данные, но сами по себе слабые, поэтому дальше требуют evidence.
context_blocks = [
    atomic_block('rule__context__np_geoname_id', 'np', 'geoname_id', family='context', min_len=2, max_block_size=1000),
    atomic_block('rule__context__np_subdivision', 'np', 'subdivision_1_iso_code', family='context', min_len=2, max_block_size=1000),
    atomic_block('rule__context__rt_geoid', 'rt', 'geoid', family='context', min_len=2, max_block_size=1000),
    atomic_block('rule__context__rt_geoname', 'rt', 'geoname', family='context', min_len=2, max_block_size=1000),
]

# Behavior rules: поведенческие fs/site-id признаки.
# Их оставляем потому, что редкие site-id часто заметно сужают область поиска.
behavior_features = [
    'source_site_365',
    'source_site_30',
    'visited_30',
    'visited_365',
    'has_account',
    'has_click_365',
    'has_click_30',
    'has_accept_365',
    'has_accept_30',
    'has_order_365',
    'has_order_30',
    'has_view_90',
]

behavior_blocks = [
    atomic_block(
        f'rule__behavior__fs_{feature}',
        'fs',
        feature,
        family='behavior',
        min_len=1,
        max_block_size=1000,
    )
    for feature in behavior_features
]

# Behavior + context: география плюс поведение.
# Такие композиты обычно чище одиночных context/behavior правил.
behavior_context_features = [
    'source_site_365',
    'has_account',
    'has_click_365',
    'has_accept_365',
    'visited_30',
]

behavior_context_blocks = [
    composite_block(
        f'rule__behavior_context__np_geoname_id__fs_{feature}',
        [('np', 'geoname_id', 'geo', None, 2), ('fs', feature, 'fs_value', None, 1)],
        family='behavior_context',
        max_block_size=1000,
    )
    for feature in behavior_context_features
]
print(behavior_context_blocks[0])

# Узкие behavior_context_device правила добавлены после rule discovery/deep-dive.
# Берём только несколько проверенных fs-признаков, чтобы не раздувать шум.
behavior_context_device_features = [
    'source_site_365',
    'has_click_365',
    'has_accept_365',
]

behavior_context_device_blocks = [
    composite_block(
        f'rule__behavior_context_device__np_subdivision__fs_{feature}__np_osfamily',
        [('np', 'subdivision_1_iso_code', 'subdivision', None, 2), ('fs', feature, 'fs_value', None, 1), ('np', 'osfamily', 'osfamily', None, 1)],
        family='behavior_context_device',
        max_block_size=1000,
    )
    for feature in behavior_context_device_features
]

identity_rescue_blocks = [
    # Identity слишком разрежен для основы blocking. Оставляем только телефон как точечный high-precision ключ.
    atomic_block('rule__identity_rescue__phone_digits', 'identity', 'phone', family='identity_rescue', min_len=7, max_block_size=1000),
    atomic_block('rule__identity_rescue__phone_prefix6', 'identity', 'phone', family='identity_rescue', transform=prefix6, min_len=6, max_block_size=1000),
]

# Coverage compound: слабый слой для покрытия профилей, которые плохо ловятся основными правилами.
# Эти правила полезны для recall, но в graph/model layer не должны быть единственным сильным сигналом.
coverage_compound_blocks = [
    composite_block(
        'rule__coverage__np_geoname_id__np_device__np_osfamily',
        [('np', 'geoname_id', 'geo', None, 2), ('np', 'device', 'device', None, 1), ('np', 'osfamily', 'osfamily', None, 1)],
        family='coverage_compound',
        max_block_size=1000,
    ),
    composite_block(
        'rule__coverage__np_subdivision__np_device__np_osfamily',
        [('np', 'subdivision_1_iso_code', 'subdivision', None, 2), ('np', 'device', 'device', None, 1), ('np', 'osfamily', 'osfamily', None, 1)],
        family='coverage_compound',
        max_block_size=1000,
    ),
    composite_block(
        'rule__coverage__email_domain__np_device__np_osfamily__sex',
        [('derived', 'email_domain', 'email_domain', None, 1), ('np', 'device', 'device', None, 1), ('np', 'osfamily', 'osfamily', None, 1), ('identity', 'sex', 'sex', None, 1)],
        family='coverage_compound',
        max_block_size=1000,
    ),
    composite_block(
        'rule__coverage__email_domain__email_initial2',
        [('derived', 'email_domain', 'email_domain', None, 1), ('derived', 'email_initial2', 'email_initial2', None, 1)],
        family='coverage_compound',
        max_block_size=1000,
    ),
]

coverage_fallback_blocks = [
    # Последний технический fallback для 100% покрытия профилей. Это не сильный ER-сигнал.
    atomic_block(
        'rule__coverage_fallback__email_hash_bucket_1024',
        'derived',
        'email_hash_bucket_1024',
        family='coverage_fallback',
        min_len=1,
        max_block_size=1000,
    ),
]

blocks = context_blocks + behavior_blocks + behavior_context_blocks + behavior_context_device_blocks + identity_rescue_blocks + coverage_compound_blocks + coverage_fallback_blocks

blocking_index = (
    pd.concat([b for b in blocks if not b.empty], ignore_index=True)
    .drop_duplicates()
    .merge(profile_core[['profile_id', 'entity_id']], on='profile_id', how='left')
)

blocking_index.to_parquet(OUTPUT_DIR / 'blocking_index.parquet', index=False)

print('blocking_index shape:', blocking_index.shape)
print('blocking families:', blocking_index['block_family'].value_counts().to_dict())
blocking_index.head(5)


                                 profile_id      block_family                                         block_rule   block_value
0      00036d8f-9320-39b0-96e9-f0a457fc82a4  behavior_context  rule__behavior_context__np_geoname_id__fs_sour...   538560|2953
1      00036d8f-9320-39b0-96e9-f0a457fc82a4  behavior_context  rule__behavior_context__np_geoname_id__fs_sour...   538560|4021
2      00036d8f-9320-39b0-96e9-f0a457fc82a4  behavior_context  rule__behavior_context__np_geoname_id__fs_sour...   538560|4129
3      0006ed5c-4676-33f0-9a75-7ee815340994  behavior_context  rule__behavior_context__np_geoname_id__fs_sour...  2025339|3052
4      0006ed5c-4676-33f0-9a75-7ee815340994  behavior_context  rule__behavior_context__np_geoname_id__fs_sour...  2025339|3772
...                                     ...               ...                                                ...           ...
51587  fffe0062-571d-350d-9cc5-257426adb19d  behavior_context  rule__behavior_context__np_geoname_id__fs_sour..

,profile_id,block_family,block_rule,block_value,entity_id
0,000163e0-b9b4-3323-84ea-6495df7fe895,context,rule__context__np_geoname_id,625144,266eb686284a35db45a797ef75c01fc276e71fee1eb2aa...
1,00036d8f-9320-39b0-96e9-f0a457fc82a4,context,rule__context__np_geoname_id,538560,5b3e5b68690777e70e9f394d45ebb85aeb5f12c68b2f06...
2,0006ed5c-4676-33f0-9a75-7ee815340994,context,rule__context__np_geoname_id,2025339,a31a963097b10e0deef187432cf215161188117d71fb9d...
3,00074a63-2a25-352d-9055-37d6b76fd76a,context,rule__context__np_geoname_id,2023469,61cc85dc1a289453bc9224085024543e68cd8054a02153...
4,0007e583-6cb5-3372-9a47-b51fde41c310,context,rule__context__np_geoname_id,499099,ba9ffba29dd3b8712929dcb596b811d519fc3313faf9fd...


## 10. Отчёт по размерам блоков

Техническая проверка размера блоков после отсечения слишком широких значений:

- сколько блоков даёт правило;
- какой максимальный размер блока;
- сколько candidate pairs получится внутри блоков;
- сколько профилей покрывает правило.

Дальше в разделе диагностики считаем `positive_recall` и `positive_pairs_captured`.


- `n_blocks` — сколько разных блоков создало правило. Например, если после фильтрации осталось 1118 разных `geoname_id`, то `n_blocks = 1118`.
- `max_block_size` — размер самого большого оставшегося `block_value`. `BLOCK_MAX_SIZE = 1000` — это верхняя граница для каждого отдельного `block_value`, а не лимит на `n_blocks`.
- `candidate_pairs` — сколько пар профилей создаёт правило внутри всех своих блоков. Для одного блока: `block_size * (block_size - 1) / 2`; затем суммируем по всем блокам.
- `n_profiles_covered` — сколько уникальных профилей попало хотя бы в один блок этого правила.
- `profile_coverage` — доля всех профилей, покрытых правилом: `n_profiles_covered / total_profiles`.


In [21]:
block_sizes = (
    blocking_index
    .groupby(['block_family', 'block_rule', 'block_value'], sort=False)
    .agg(
        block_size=('profile_id', 'nunique'),
        entity_nunique=('entity_id', 'nunique'),
    )
    .reset_index()
)
block_sizes['candidate_pairs'] = (block_sizes['block_size'] * (block_sizes['block_size'] - 1) // 2).astype('int64')

blocking_rule_size_report = (
    block_sizes
    .groupby(['block_family', 'block_rule'], sort=False)
    .agg(
        n_blocks=('block_value', 'size'),
        max_block_size=('block_size', 'max'),
        candidate_pairs=('candidate_pairs', 'sum'),
    )
    .reset_index()
)

rule_coverage = (
    blocking_index
    .groupby(['block_family', 'block_rule'])['profile_id']
    .nunique()
    .rename('n_profiles_covered')
    .reset_index()
)

n_profiles = raw['profile_id'].nunique()
total_possible_pairs = int(n_profiles * (n_profiles - 1) // 2)

blocking_rule_size_report = blocking_rule_size_report.merge(
    rule_coverage,
    on=['block_family', 'block_rule'],
    how='left',
)
blocking_rule_size_report['profile_coverage'] = blocking_rule_size_report['n_profiles_covered'] / n_profiles
blocking_rule_size_report = blocking_rule_size_report.sort_values(['block_family', 'candidate_pairs', 'max_block_size'])

blocking_rule_size_report.to_parquet(OUTPUT_DIR / 'blocking_rule_size_report.parquet', index=False)
blocking_rule_size_report.to_csv(OUTPUT_DIR / 'blocking_rule_size_report.csv', index=False)

blocking_rule_size_report.head(10)


,block_family,block_rule,n_blocks,max_block_size,candidate_pairs,n_profiles_covered,profile_coverage
14,behavior,rule__behavior__fs_has_order_30,83,751,352940,2034,0.032845
13,behavior,rule__behavior__fs_has_order_365,152,972,1189268,5629,0.090897
10,behavior,rule__behavior__fs_has_click_30,335,754,1820519,7769,0.125454
12,behavior,rule__behavior__fs_has_accept_30,220,831,1828901,7130,0.115136
7,behavior,rule__behavior__fs_visited_365,100,898,1915802,6005,0.096969
5,behavior,rule__behavior__fs_source_site_30,184,853,2557943,11796,0.190482
6,behavior,rule__behavior__fs_visited_30,283,990,3784535,10221,0.165049
9,behavior,rule__behavior__fs_has_click_365,570,744,4324067,14179,0.228963
4,behavior,rule__behavior__fs_source_site_365,272,963,6188234,16793,0.271174
8,behavior,rule__behavior__fs_has_account,638,974,14167865,14023,0.226444


## 11. Quality checks витрины

Эти проверки оценивают не качество ER-модели, а техническую консистентность витрины: гранулярность, дубли, сохранённые артефакты, согласованность long/stats слоёв и целостность `blocking_index`.


In [22]:
expected_files = [
    'profile_core.parquet',
    'profile_value_summary_long.parquet',
    'profile_feature_stats_long.parquet',
    'blocking_index.parquet',
    'blocking_rule_size_report.parquet',
    'blocking_rule_size_report.csv',
    'feature_dictionary.csv',
]
if SAVE_EVENT_LEVEL_LONG:
    expected_files.append('profile_event_value_long.parquet')

qc_rows = []

def add_check(name, passed, metric=None, expected=None, severity='fail'):
    qc_rows.append({
        'check_name': name,
        'status': 'ok' if bool(passed) else severity,
        'metric': metric,
        'expected': expected,
    })

add_check(
    'raw_profile_id_not_null',
    raw['profile_id'].notna().all(),
    int(raw['profile_id'].isna().sum()),
    '0 null profile_id',
)
add_check(
    'raw_entity_id_not_null',
    raw['entity_id'].notna().all(),
    int(raw['entity_id'].isna().sum()),
    '0 null entity_id',
)
add_check(
    'profile_core_one_row_per_profile',
    profile_core['profile_id'].is_unique,
    profile_core['profile_id'].duplicated().sum(),
    'no duplicate profile_id',
)
add_check(
    'profile_core_matches_raw_profiles',
    profile_core['profile_id'].nunique() == raw['profile_id'].nunique(),
    profile_core['profile_id'].nunique(),
    raw['profile_id'].nunique(),
)
add_check(
    'profile_maps_to_single_entity',
    profile_core['entity_id_nunique'].max() <= 1,
    int(profile_core['entity_id_nunique'].max()),
    '<= 1',
)
add_check(
    'value_summary_no_duplicate_profile_feature_value',
    not profile_value_summary_long.duplicated(['profile_id', 'feature_key', 'value_norm']).any(),
    int(profile_value_summary_long.duplicated(['profile_id', 'feature_key', 'value_norm']).sum()),
    '0 duplicates',
)
primary_counts = profile_value_summary_long.groupby(['profile_id', 'feature_key'], observed=True)['is_primary_value'].sum()
add_check(
    'one_primary_value_per_profile_feature',
    primary_counts.eq(1).all(),
    int((~primary_counts.eq(1)).sum()),
    'every profile-feature has exactly 1 primary value',
)
rank_min = profile_value_summary_long.groupby(['profile_id', 'feature_key'], observed=True)['value_rank'].min()
add_check(
    'value_rank_starts_at_one',
    rank_min.eq(1).all(),
    int((~rank_min.eq(1)).sum()),
    'every profile-feature rank starts at 1',
)
feature_count_recon = (
    profile_value_summary_long
    .groupby(['profile_id', 'feature_key'], observed=True)['value_count']
    .sum()
    .reset_index()
    .merge(
        profile_feature_stats_long[['profile_id', 'feature_key', 'feature_value_count']],
        on=['profile_id', 'feature_key'],
        how='outer',
    )
)
feature_count_recon['delta'] = feature_count_recon['value_count'].fillna(-1) - feature_count_recon['feature_value_count'].fillna(-2)
add_check(
    'value_summary_matches_feature_stats_counts',
    feature_count_recon['delta'].eq(0).all(),
    int((~feature_count_recon['delta'].eq(0)).sum()),
    '0 mismatched profile-feature counts',
)

add_check(
    'feature_event_coverage_in_0_1',
    profile_feature_stats_long['feature_event_coverage'].between(0, 1).all(),
    profile_feature_stats_long['feature_event_coverage'].max(),
    '<= 1',
)
add_check(
    'feature_observations_per_event_ge_event_coverage',
    (profile_feature_stats_long['feature_observations_per_event'] >= profile_feature_stats_long['feature_event_coverage']).all(),
    int((profile_feature_stats_long['feature_observations_per_event'] < profile_feature_stats_long['feature_event_coverage']).sum()),
    '0 rows where observations_per_event < event_coverage',
)

add_check(
    'blocking_index_no_duplicate_mapping',
    not blocking_index.duplicated(['profile_id', 'block_rule', 'block_value']).any(),
    int(blocking_index.duplicated(['profile_id', 'block_rule', 'block_value']).sum()),
    '0 duplicate mappings',
)
add_check(
    'blocking_profiles_exist_in_core',
    blocking_index['profile_id'].isin(profile_core['profile_id']).all(),
    int((~blocking_index['profile_id'].isin(profile_core['profile_id'])).sum()),
    'all blocking profiles in profile_core',
)
blocking_profile_coverage = blocking_index['profile_id'].nunique() / profile_core['profile_id'].nunique()
add_check(
    'blocking_index_covers_all_profiles',
    blocking_profile_coverage == 1.0,
    blocking_profile_coverage,
    '1.0 profile coverage',
)
missing_files = [name for name in expected_files if not (OUTPUT_DIR / name).exists()]
add_check(
    'expected_files_saved',
    len(missing_files) == 0,
    ', '.join(missing_files) if missing_files else 'none',
    'all expected files exist',
)

quality_check_report = pd.DataFrame(qc_rows)
quality_check_report.to_csv(OUTPUT_DIR / 'quality_check_report.csv', index=False)
quality_check_report


,check_name,status,metric,expected
0,raw_profile_id_not_null,ok,0,0 null profile_id
1,raw_entity_id_not_null,ok,0,0 null entity_id
2,profile_core_one_row_per_profile,ok,0,no duplicate profile_id
3,profile_core_matches_raw_profiles,ok,61927,61927
4,profile_maps_to_single_entity,ok,1,<= 1
5,value_summary_no_duplicate_profile_feature_value,ok,0,0 duplicates
6,one_primary_value_per_profile_feature,ok,0,every profile-feature has exactly 1 primary value
7,value_rank_starts_at_one,ok,0,every profile-feature rank starts at 1
8,value_summary_matches_feature_stats_counts,ok,0,0 mismatched profile-feature counts
9,feature_event_coverage_in_0_1,ok,1.0,<= 1


## 12. Диагностика `blocking_index`

Одного размера блока недостаточно. У хорошего blocking-правила должны одновременно быть:

- `positive_recall` — какую долю размеченных positive pairs правило может поймать;
- контролируемый `candidate_pairs`, чтобы не передать дальше слишком много пар;
- контролируемый `max_block_size`, чтобы не создавать гигантские шумные блоки.

Методика расчёта `positive_recall`:

1. По `entity_id` строим все настоящие positive pairs: если два разных `profile_id` имеют один `entity_id`, это размеченная positive-пара.
2. Для каждого blocking-правила проверяем, попали ли оба профиля из positive-пары в один и тот же блок, то есть имеют одинаковые `block_family`, `block_rule`, `block_value`.
3. Считаем `positive_pairs_captured` — сколько размеченных positive pairs правило поймало.
4. Считаем `positive_recall = positive_pairs_captured / total_positive_pairs`.

Пример.
```text
Допустим, в ground truth есть 3 клиента с дублями:
entity_id = E1: P1, P2
entity_id = E2: P3, P4
entity_id = E3: P5, P6
Значит, настоящие positive pairs:
(P1, P2)
(P3, P4)
(P5, P6)
Всего:
total_positive_pairs = 3

Теперь проверяем конкретное правило:
rule__context__np_geoname_id
Оно построило такие блоки:
block_value = 524901: P1, P2, P9
block_value = 2013348: P3, P8
block_value = 1496747: P5, P6
Смотрим, какие настоящие пары попали в один и тот же блок:
(P1, P2) попали вместе в block_value = 524901 → поймали
(P3, P4) не попали вместе: P3 есть, P4 нет → не поймали
(P5, P6) попали вместе в block_value = 1496747 → поймали
Получается:
positive_pairs_captured = 2
total_positive_pairs = 3
positive_recall = 2 / 3 = 66.7%
```

Важно, positive_recall считается уже после ограничения размера блоков. Это правильно для практической оценки, потому что нас интересует не теоретический recall признака, а recall реально используемого blocking-правила с ограничениями.
Почему для blocking важен именно recall: если настоящая пара не попала в candidate pairs на этапе blocking, модель сравнения дальше её уже не увидит. Поэтому blocking сначала должен не потерять как можно больше настоящих дублей, а уже затем мы контролируем объём через `candidate_pairs` и `max_block_size`.

`entity_id` ниже используется только для оценки. Его нельзя использовать как production-признак.


In [23]:
from itertools import combinations

positive_pair_rows = []
for entity_id, grp in profile_core.groupby('entity_id', sort=False):
    profiles = sorted(map(str, grp['profile_id'].dropna().unique()))
    if len(profiles) < 2:
        continue
    for left_profile_id, right_profile_id in combinations(profiles, 2):
        positive_pair_rows.append((len(positive_pair_rows), entity_id, left_profile_id, right_profile_id))

positive_pairs = pd.DataFrame(
    positive_pair_rows,
    columns=['pair_id', 'entity_id', 'profile_id_l', 'profile_id_r'],
)

total_positive_pairs = len(positive_pairs)

def captured_positive_pairs_for_index(index_df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    if positive_pairs.empty or index_df.empty:
        return pd.DataFrame(columns=['block_family', 'block_rule', 'positive_pairs_captured'])

    group_cols = ['block_family', 'block_rule']
    for (family, rule), rule_idx in index_df[['profile_id', 'block_family', 'block_rule', 'block_value']].drop_duplicates().groupby(group_cols, sort=False):
        left = positive_pairs.merge(
            rule_idx.rename(columns={'profile_id': 'profile_id_l'}),
            on='profile_id_l',
            how='inner',
        )
        captured = left.merge(
            rule_idx.rename(columns={'profile_id': 'profile_id_r'}),
            on=['profile_id_r', 'block_value'],
            how='inner',
            suffixes=('', '_r'),
        )
        rows.append((family, rule, int(captured['pair_id'].nunique())))

    return pd.DataFrame(rows, columns=['block_family', 'block_rule', 'positive_pairs_captured'])

positive_capture = captured_positive_pairs_for_index(blocking_index)

blocking_rule_quality_report = blocking_rule_size_report.merge(
    positive_capture,
    on=['block_family', 'block_rule'],
    how='left',
)
blocking_rule_quality_report['positive_pairs_captured'] = blocking_rule_quality_report['positive_pairs_captured'].fillna(0).astype('int64')
blocking_rule_quality_report['total_positive_pairs'] = total_positive_pairs
blocking_rule_quality_report['positive_recall'] = np.where(
    total_positive_pairs > 0,
    blocking_rule_quality_report['positive_pairs_captured'] / total_positive_pairs,
    np.nan,
)
blocking_rule_quality_report['pairs_per_positive_captured'] = np.where(
    blocking_rule_quality_report['positive_pairs_captured'] > 0,
    blocking_rule_quality_report['candidate_pairs'] / blocking_rule_quality_report['positive_pairs_captured'],
    np.inf,
)
blocking_rule_quality_report['recommended_for_next_step'] = (
    blocking_rule_quality_report['candidate_pairs'].le(10_000_000)
    & blocking_rule_quality_report['max_block_size'].le(BLOCK_MAX_SIZE)
    & ~blocking_rule_quality_report['block_family'].eq('identity_rescue')
)

blocking_rule_quality_report = blocking_rule_quality_report[[
    'block_family',
    'block_rule',
    'positive_recall',
    'positive_pairs_captured',
    'total_positive_pairs',
    'candidate_pairs',
    'pairs_per_positive_captured',
    'max_block_size',
    'n_blocks',
    'n_profiles_covered',
    'profile_coverage',
    'recommended_for_next_step',
]].sort_values(['positive_recall', 'candidate_pairs'], ascending=[False, True])

blocking_rule_quality_report.to_parquet(OUTPUT_DIR / 'blocking_rule_quality_report.parquet', index=False)
blocking_rule_quality_report.to_csv(OUTPUT_DIR / 'blocking_rule_quality_report.csv', index=False)

recommended_blocking_rules = blocking_rule_quality_report[
    blocking_rule_quality_report['recommended_for_next_step']
].copy()
recommended_blocking_rules.to_csv(OUTPUT_DIR / 'recommended_blocking_rules.csv', index=False)

recommended_rule_keys = set(
    map(tuple, recommended_blocking_rules[['block_family', 'block_rule']].to_numpy())
)
recommended_index = blocking_index[
    pd.MultiIndex.from_frame(blocking_index[['block_family', 'block_rule']]).isin(recommended_rule_keys)
]
blocking_profile_coverage_report = pd.DataFrame([
    {
        'index_scope': 'all_blocking_index',
        'n_profiles_covered': blocking_index['profile_id'].nunique(),
        'total_profiles': profile_core['profile_id'].nunique(),
        'profile_coverage': blocking_index['profile_id'].nunique() / profile_core['profile_id'].nunique(),
    },
    {
        'index_scope': 'recommended_for_next_step',
        'n_profiles_covered': recommended_index['profile_id'].nunique(),
        'total_profiles': profile_core['profile_id'].nunique(),
        'profile_coverage': recommended_index['profile_id'].nunique() / profile_core['profile_id'].nunique(),
    },
])
blocking_profile_coverage_report.to_csv(OUTPUT_DIR / 'blocking_profile_coverage_report.csv', index=False)

blocking_rule_quality_report.head(5)


,block_family,block_rule,positive_recall,positive_pairs_captured,total_positive_pairs,candidate_pairs,pairs_per_positive_captured,max_block_size,n_blocks,n_profiles_covered,profile_coverage,recommended_for_next_step
27,coverage_compound,rule__coverage__np_subdivision__np_device__np_...,0.760191,12439,16363,7758595,623.731409,947,690,45453,0.733977,True
26,coverage_compound,rule__coverage__np_geoname_id__np_device__np_o...,0.752613,12315,16363,5838365,474.085668,971,1831,44843,0.724127,True
22,context,rule__context__rt_geoname,0.723461,11838,16363,6090176,514.459875,980,1052,37405,0.604018,True
21,context,rule__context__np_geoname_id,0.719795,11778,16363,5688766,482.999321,980,1118,36865,0.595298,True
20,context,rule__context__rt_geoid,0.715700,11711,16363,5673536,484.462130,980,1054,36343,0.586868,True


## 13. Финальный чеклист артефактов

После выполнения ноутбук должен сохранить базовые слои витрины, диагностические отчёты и отчёты по blocking.

Базовые слои:

```text
profile_core.parquet
profile_value_summary_long.parquet
profile_feature_stats_long.parquet
blocking_index.parquet
feature_dictionary.csv
```

Quality checks:

```text
quality_check_report.csv
```

Диагностика blocking:

```text
blocking_rule_size_report.parquet
blocking_rule_size_report.csv
blocking_rule_quality_report.parquet
blocking_rule_quality_report.csv
recommended_blocking_rules.csv
```

Опционально, если `SAVE_EVENT_LEVEL_LONG=True`:

```text
profile_event_value_long.parquet
```

Методология:

```text
profile_value_summary_long — источник истины для multi-value признаков.
blocking_index — many-to-many индекс для генерации candidate pairs.
blocking_rule_quality_report — первый recall/reduction фильтр для blocking-правил.
```


In [24]:
print('Saved files:')
for path in sorted(OUTPUT_DIR.glob('*')):
    print('-', path.name)

print('\nCore shapes:')
print('profile_core:', profile_core.shape)
print('profile_value_summary_long:', profile_value_summary_long.shape)
print('profile_feature_stats_long:', profile_feature_stats_long.shape)
print('blocking_index:', blocking_index.shape)

print('\nQuality checks:')
display(quality_check_report)

print('\nBlocking positive recall:')
display(blocking_rule_quality_report)


Saved files:
- blocking_index.parquet
- blocking_profile_coverage_report.csv
- blocking_rule_quality_report.csv
- blocking_rule_quality_report.parquet
- blocking_rule_size_report.csv
- blocking_rule_size_report.parquet
- feature_dictionary.csv
- profile_core.parquet
- profile_event_value_long.parquet
- profile_feature_stats_long.parquet
- profile_value_summary_long.parquet
- quality_check_report.csv
- recommended_blocking_rules.csv

Core shapes:
profile_core: (61927, 8)
profile_value_summary_long: (1844644, 13)
profile_feature_stats_long: (1199575, 15)
blocking_index: (1300577, 5)

Quality checks:


,check_name,status,metric,expected
0,raw_profile_id_not_null,ok,0,0 null profile_id
1,raw_entity_id_not_null,ok,0,0 null entity_id
2,profile_core_one_row_per_profile,ok,0,no duplicate profile_id
3,profile_core_matches_raw_profiles,ok,61927,61927
4,profile_maps_to_single_entity,ok,1,<= 1
5,value_summary_no_duplicate_profile_feature_value,ok,0,0 duplicates
6,one_primary_value_per_profile_feature,ok,0,every profile-feature has exactly 1 primary value
7,value_rank_starts_at_one,ok,0,every profile-feature rank starts at 1
8,value_summary_matches_feature_stats_counts,ok,0,0 mismatched profile-feature counts
9,feature_event_coverage_in_0_1,ok,1.0,<= 1



Blocking positive recall:


,block_family,block_rule,positive_recall,positive_pairs_captured,total_positive_pairs,candidate_pairs,pairs_per_positive_captured,max_block_size,n_blocks,n_profiles_covered,profile_coverage,recommended_for_next_step
27,coverage_compound,rule__coverage__np_subdivision__np_device__np_...,0.760191,12439,16363,7758595,623.731409,947,690,45453,0.733977,True
26,coverage_compound,rule__coverage__np_geoname_id__np_device__np_o...,0.752613,12315,16363,5838365,474.085668,971,1831,44843,0.724127,True
22,context,rule__context__rt_geoname,0.723461,11838,16363,6090176,514.459875,980,1052,37405,0.604018,True
21,context,rule__context__np_geoname_id,0.719795,11778,16363,5688766,482.999321,980,1118,36865,0.595298,True
20,context,rule__context__rt_geoid,0.715700,11711,16363,5673536,484.462130,980,1054,36343,0.586868,True
23,context,rule__context__np_subdivision,0.699322,11443,16363,7350865,642.389671,977,249,32961,0.532256,True
25,coverage_compound,rule__coverage__email_domain__np_device__np_os...,0.471002,7707,16363,3024203,392.396912,967,313,14245,0.230029,True
18,behavior_context_device,rule__behavior_context_device__np_subdivision_...,0.195319,3196,16363,1009856,315.974969,240,7488,25612,0.413584,True
13,behavior_context,rule__behavior_context__np_geoname_id__fs_sour...,0.187374,3066,16363,786992,256.683627,269,7046,23200,0.374635,True
17,behavior_context_device,rule__behavior_context_device__np_subdivision_...,0.173685,2842,16363,769231,270.665376,184,8692,21362,0.344955,True
